# SSA: Parameter Exploration

### Libruary imports

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np

from src import ssa_visualization as viz
from src.ssa_simulation import compute_sample_moments, simulate_telegraph
from src.ssa_visualization import show_sample_moments, show_single_trajectory

In [2]:
def run_experiment(
    title: str, n_grid=1000, time_step=None, t_end=None,
    show_single_trajectory=False, **kwargs
):
    """Run the Gillespie SSA simulation and plot the resulting sample moments.

    title : str
        The descriptive name of the parameter exploration setup.
    n_grid, time_step, t_end :
        Real-time resampling controls forwarded to compute_sample_moments.
        Use time_step (seconds) for a fixed step, or n_grid for a point count.
    show_single_trajectory : bool
        If True, also show a separate standalone figure of one raw realization
        (gene state, RNA count, and its own deviation product).
    **kwargs : dict
        Model parameters to dynamically override.
    """
    base_params = {
        "k_on": 0.1,
        "k_off": 0.1,
        "k_syn": 10.0,
        "k_deg": 1,
        "t0": 0.0,
        "g0": 0,
        "r0": 0,
        "n_sim": 4000,
        "n_rep": 1000
    }
    
    current_params = {**base_params, **kwargs}
    
    data = simulate_telegraph(**current_params)
    moments = compute_sample_moments(
        data, t_end=t_end, n_grid=n_grid, time_step=time_step
    )
    
    show_sample_moments(moments, title=title)
    if show_single_trajectory:
        viz.show_single_trajectory(
            data, trajectory_index=0, title=f"{title} (Single Realization)"
        )

---
### 1. Fast vs. Slow Gene Switching

The **switching regime** is set by how $k_{\text{on}}$ and $k_{\text{off}}$
compare to the mRNA degradation rate $k_{\text{deg}}$.

**Goal:** see how the promoter switching speed shapes mRNA noise.
**Watch:** width of the ⟨R⟩ band ($\sigma_R$) and the strength of the
Gene–RNA covariance.

**1a. Slow switching — single-allele baseline** ($k_{\text{on}}=1.4,\; k_{\text{off}}=4.74,\; k_{\text{syn}}=38.8$)

One locus (e.g. the *Mbnl2* maternal allele). Gene is mostly OFF ($k_{\text{on}}<k_{\text{off}}$) →
rare, distinct bursts → **wide $\sigma_R$**, **strong G–R covariance**.

In [3]:
run_experiment(
    title="Slow Gene Switching Regime",
    k_on=1.4,
    k_off=4.74,
    k_syn=38.8,
    show_single_trajectory=True
)

**1b. Fast switching — total (two-allele) expression** ($k_{\text{on}}=2.71,\; k_{\text{off}}=12.0,\; k_{\text{syn}}=89.5$)

Pooled signal from both alleles raises the effective $k_{\text{on}}$ → rapid flickering with high
synthesis on overlapping bursts → **narrow $\sigma_R$**, **near-zero covariance**.

In [4]:
run_experiment(
    title="Fast Gene Switching Regime",
    k_on=2.71,
    k_off=12.0,
    k_syn=89.5, 
    show_single_trajectory=True
)

---
## 2. High vs. Low Expression Level

Steady-state ⟨R⟩ is set by the **burst size** $k_{\text{syn}}/k_{\text{off}}$.

**Goal:** show that promoter architecture changes burst size, not burst frequency
($k_{\text{on}}$ fixed at $0.5$).

**2a. High expression — TATA + Inr promoter** ($k_{\text{on}}=0.5,\; k_{\text{off}}=4,\; k_{\text{syn}}=56$)

Stable ON state and strong synthesis → large burst size ($k_{\text{syn}}/k_{\text{off}}\approx 14$) →
**high ⟨R⟩**.

In [5]:
run_experiment(
    title="High expression",
    k_on=0.5,
    k_off=4,
    k_syn=56,
    show_single_trajectory=True
)


**2b. Low expression — TATA-less / Inr-less promoter** ($k_{\text{on}}=0.5,\; k_{\text{off}}=10,\; k_{\text{syn}}=30$)

Unstable ON state → small burst size ($k_{\text{syn}}/k_{\text{off}}\approx 3$) →
**low ⟨R⟩**, large relative noise.

In [6]:
run_experiment(
    title="Low expression",
    k_on=0.5,
    k_off=10,
    k_syn=30,
    show_single_trajectory=True
)


---
## 3. Effect of Initial Conditions

Same rates, different starting state $(g_0, r_0)$.

**Goal:** confirm all initial conditions **relax to the same steady state**;
the transient just shows how the system forgets its start.

**3a. Inactive start — idle G1 cell** ($g_0=0,\; r_0=0$)

Gene resting, no mRNA → ⟨R⟩ **builds up** from zero (pure activation kinetics).

In [7]:
run_experiment(
    title="Inactive Start (g0=0, r0=0)",
    g0=0,
    r0=0,
    show_single_trajectory=True
)

**3b. Active-burst start — smFISH at burst peak** ($g_0=1,\; r_0=15$)

Gene ON and mRNA already elevated → ⟨R⟩ **relaxes down** to the same steady state.

In [8]:
run_experiment(
    title="Active-Burst Start (g0=1, r0=15)",
    t0=0.0,
    g0=1,
    r0=15,
    show_single_trajectory=True
)

---
## 4. Effect of Numerical Parameters

These knobs change the **estimate quality**.

**Goal:** show how ensemble size and run length affect the sampled moments.

**4a. Low numerical scale — robustness test** ($n_{\text{sim}}=1000$)

Short trajectories capture only a few bursts → shows how sampling noise distorts the estimated
moments in small datasets.

In [9]:
run_experiment(
    title="Numerical Effects: Low Scale (n_sim=1000)",
    n_sim=1000,
)

**4b. Experiment scale — true statistical power** ($n_{\text{rep}}=224,\; n_{\text{sim}}=5000$)

$n_{\text{rep}}$ matches the 224 sequenced fibroblasts; long runs reach true steady state for a valid
distribution comparison.

In [10]:
run_experiment(
    title="Numerical Effects: Experiment Scale (n_rep=224, n_sim=5000)",
    n_rep=224,
    n_sim=5000,
)